In [4]:
# breaking
import numpy as np
from tqdm.auto import tqdm
from superconductivity.utilities.legacy.functions import bin_y_over_x
from superconductivity.models.mar import get_Imar_nA
from superconductivity.utilities.constants import G0_muS
from superconductivity.models.basics.noise import (
    apply_voltage_noise,
    make_bias_support_grid,
)
from superconductivity.utilities.functions.upsampling import upsample
from superconductivity.utilities.functions.binning import nanbin

data = np.load("breaking/fit_data2.npz", allow_pickle=True)

V_mV = data["V_mV"]
x_arbu = data["x_arbu"]
tau = data["tau"]
GN_G0 = data["GN_G0"]
T_K = 0.0
Delta_meV = 0.1885
gamma_meV = 1e-6
sigmaV_mV = 26e-3

V_ = V_mV / Delta_meV
Isim_nA = np.full((x_arbu.shape[0], V_mV.shape[0]), np.nan)
Isim = np.full_like(Isim_nA, np.nan)
dGsim = np.full_like(Isim_nA, np.nan)
dRsim = np.full_like(Isim_nA, np.nan)

V_support_mV = make_bias_support_grid(V_mV, sigmaV_mV)

for i, xi_arbu in enumerate(tqdm(x_arbu)):
    tempI = get_Imar_nA(
        V_mV=V_support_mV,
        tau=tau[i, :],
        T_K=T_K,
        Delta_meV=Delta_meV,
        gamma_meV=gamma_meV,
    )
    broadened_support = apply_voltage_noise(
        V_support_mV,
        tempI,
        float(sigmaV_mV),
        order=32,
    )
    Isim_nA[i, :] = np.interp(
        V_mV,
        V_support_mV,
        broadened_support,
    )

    Isim[i, :] = Isim_nA[i, :] / (Delta_meV * G0_muS * GN_G0[i])
    dGsim[i, :] = np.gradient(Isim[i, :], V_, axis=-1)

    mask = np.isfinite(Isim[i, :])
    if np.sum(mask):
        Vsim = nanbin(
            upsample(V_[mask], axis=-1),
            upsample(Isim[i, mask], axis=-1),
            V_,
        )
        dRsim[i, :] = np.gradient(Vsim, V_)

np.savez(
    "breaking/sim.npz",
    Vbias_mV=V_mV,
    Vbias=V_,
    x_arbu=x_arbu,
    Isim_nA=Isim_nA,
    Isim=Isim,
    dGsim=dGsim,
    dRsim=dRsim,
    tau=tau,
    GN_G0=GN_G0,
    T_K=T_K,
    Delta_meV=Delta_meV,
    gamma_meV=gamma_meV,
    sigmaV_mV=sigmaV_mV,
)

  0%|          | 0/268 [00:00<?, ?it/s]

/var/folders/kc/8fnzl3f94vxgl8w4wm3wfvk80000gn/T/ipykernel_3241/2160927895.py:53: RuntimeWarning: invalid value encountered in divide
  Isim[i, :] = Isim_nA[i, :] / (Delta_meV * G0_muS * GN_G0[i])
